# Endpoint de inferencia en Google Colab (respaldo de RunPod)
Levanta el mismo `inferencia/app.py` en la GPU de Colab y lo publica con un túnel de Cloudflare (la URL `https://….trycloudflare.com` corre en la nube de Google, no en tu PC).
Copia la URL en `INFERENCE_URL` de Render. **Mantén esta pestaña abierta** durante la demo.

In [ ]:
!git clone -q https://github.com/MauleyArias/sistema_de_control_de_peaje.git repo
%cd repo/inferencia
!pip install -q -r requirements.txt
!python diagnostico.py | tee diagnostico_colab.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
import os; os.makedirs(SALIDA, exist_ok=True)

import shutil
for f in ['m2_seg.pt', 'm3_pose.pt']:
    if os.path.exists(f'{SALIDA}/{f}'):
        shutil.copy(f'{SALIDA}/{f}', f'modelos/{f}')
print(os.listdir('modelos'))

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ['INFERENCE_API_KEY'] = userdata.get('INFERENCE_API_KEY')   # mismo valor que en Render
os.environ['DISPOSITIVO'] = '0'
os.environ['PROVEEDOR'] = 'colab'
servidor = subprocess.Popen('uvicorn app:app --host 0.0.0.0 --port 8000 > /content/uvicorn.log 2>&1', shell=True)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunel = subprocess.Popen('./cloudflared tunnel --url http://localhost:8000 > /content/tunel.log 2>&1', shell=True)
import time, re; time.sleep(8)
print(re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunel.log').read()))

In [ ]:
!tail -n 30 /content/uvicorn.log